In [3]:
import geopandas as gpd
import osmnx as ox
import polyline
import yaml
from shapely.geometry import Polygon, LineString
import folium
from lib.strava import Strava


In [25]:
# Read configuration files
secrets = yaml.safe_load(open('conf/secrets.yaml'))

# Log into Strava
strava = Strava(secrets['strava'])
strava.login()

# 1. Define the target area using your coordinates (Lon, Lat)
# Generated with https://www.keene.edu/campus/maps/tool/
coords = [
    (5.3972580, 43.2077929),
    (5.4035541, 43.2158312),
    (5.4066771, 43.2172421),
    (5.4050257, 43.2208042),
    (5.3960657, 43.2298034),
    (5.3980870, 43.2356715),
    (5.3858479, 43.2365669),
    (5.3866594, 43.2348639),
    (5.3855548, 43.2328185),
    (5.3818907, 43.2310732),
    (5.3804925, 43.2266032),
    (5.3788414, 43.2253853),
    (5.3744430, 43.2275809),
    (5.3707077, 43.2271064),
    (5.3682018, 43.2282935),
    (5.3658158, 43.2271114),
    (5.3637129, 43.2277368),
    (5.3611230, 43.2310688),
    (5.3566167, 43.2306449),
    (5.3532564, 43.2271955),
    (5.3472238, 43.2245628),
    (5.3460100, 43.2210584),
    (5.3473629, 43.2161735),
    (5.3499494, 43.2151176),
    (5.3498271, 43.2134514),
    (5.3446873, 43.2145093),
    (5.3427697, 43.2174135),
    (5.3346106, 43.2147142),
    (5.3474018, 43.2084765),
    (5.3521665, 43.2127833),
    (5.3547708, 43.2135737),
    (5.3553627, 43.2129211),
    (5.3510300, 43.2066459),
    (5.3972580, 43.2077929),

]
calanques_poly = Polygon(coords)


In [26]:
# ==========================================
# STEP 1: Fetch OSM Trails
# ==========================================
print("Fetching OSM trails...")
# We filter for paths, footways, and tracks
tags = {'highway': ['path', 'footway', 'track']}

# Fetch features inside the polygon
osm_trails = ox.features.features_from_polygon(calanques_poly, tags=tags)

# Filter out polygons (like plazas) to only keep LineStrings (the actual trails)
osm_trails = osm_trails[osm_trails.geometry.type.isin(['LineString', 'MultiLineString'])]

# Convert to a projected Coordinate Reference System (EPSG:2154 is for France)
# This is CRITICAL so we can calculate distances and buffers in meters, not degrees!
osm_trails = osm_trails.to_crs(epsg=2154)

print(f"Found {len(osm_trails)} trails")


Fetching OSM trails...
Found 385 trails


In [27]:
# Convert back to WGS84 (EPSG:4326) for folium
osm_trails_wgs84 = osm_trails.to_crs(epsg=4326)

# Calculate center of the area
bounds = osm_trails_wgs84.total_bounds  # [minx, miny, maxx, maxy]
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

# Create folium map
m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles='OpenStreetMap')

# Add the polygon boundary
folium.Polygon(
    locations=[(lat, lon) for lon, lat in coords],
    color='blue',
    fill=True,
    fillOpacity=0.1,
    weight=2,
    popup='Search Area'
).add_to(m)

# Add each trail to the map
for idx, row in osm_trails_wgs84.iterrows():
    geom = row.geometry
    if geom.geom_type == 'LineString':
        coords_list = [(lat, lon) for lon, lat in geom.coords]
        folium.PolyLine(
            coords_list,
            color='red',
            weight=2,
            opacity=0.7,
            popup=f"Trail {idx}"
        ).add_to(m)
    elif geom.geom_type == 'MultiLineString':
        for line in geom.geoms:
            coords_list = [(lat, lon) for lon, lat in line.coords]
            folium.PolyLine(
                coords_list,
                color='red',
                weight=2,
                opacity=0.7,
                popup=f"Trail {idx}"
            ).add_to(m)

# Display the map
m
